In [1]:
n_components_list = [2, 3, 5] # number of components in each model we are going to use to compute WBIC and FE

# wbic mcmc settings
n_tune=2500
n_draws=1000
n_chains=1

# wbic block settings
import os
# parallel_n_jobs=os.cpu_count()
parallel_n_jobs=4
parallel_verbose=10

In [2]:
regimes = [50, 250, 5000]
print(f"Running on a machine with {os.cpu_count()} cpu cores")

Running on a machine with 8 cpu cores


In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
from pathlib import Path
from os import environ
import os

ROOT = Path(f"{Path.cwd().parents[1]}/outputs")
basedir = Path(f"{ROOT}/mixpoisson")
datadir = Path(f"{basedir}/data")
outputdir = Path(f"{basedir}/wbic")

if not outputdir.exists():
  outputdir.mkdir(exist_ok=True)
  print(f"Created {outputdir}!")

print(f"Using datadir={datadir}")
print(f"Using outputdir={outputdir}")

dry = False # whether or not to save results

Using datadir=/Users/ashrafahmed/workspace/waterloo-slt-reading-group/zoo/python/projects/luna/outputs/mixpoisson/data
Using outputdir=/Users/ashrafahmed/workspace/waterloo-slt-reading-group/zoo/python/projects/luna/outputs/mixpoisson/wbic


In [5]:
import pandas as pd

dgps_file = f"{datadir}/dgp.csv"
dgps=pd.read_csv(dgps_file, index_col=0).reset_index()
dgps.head()

,dsid,m0,m1,w0,w1
0,regular,15,1,0.75,0.25
1,e-singular,10,5,0.75,0.25
2,singular1,10,5,1.00,0.00
3,singular2,5,5,0.75,0.25


In [6]:
from scipy_extensions import mixpoisson
import numpy as np


def find_truth_by_dsid(dsid: str):
  dgp = dgps.query(f"dsid=='{dsid}'")
  truth = dgp[["m0", "m1", "w0", "w1"]].iloc[0].tolist()
  return truth


def n_components_by_dsid(dsid: str):
  truth = find_truth_by_dsid(dsid)
  half_space = int(np.ceil(len(truth)/2))
  mus = truth[0: half_space]
  weights = truth[half_space: len(truth)]
  return mixpoisson.n_components(mus=mus, weights=weights)


def rlct_by_dsid(dsid: str, k: int):
  n_components0 = n_components_by_dsid(dsid)
  return mixpoisson.rlct(d=1, r=n_components0, k=k)


def afe_by_dsid(dsid: str, X, k):
  truth = find_truth_by_dsid(dsid)
  rlct = rlct_by_dsid(dsid, k=k)

  half_space = int(np.ceil(len(truth)/2))
  mus0 = truth[0: half_space]
  weights0 = truth[half_space: len(truth)]
  return mixpoisson.afe(X=X, mus=mus0, weights=weights0, rlct=rlct)

In [7]:
rlct_by_dsid("singular1", k=2)

0.75

In [8]:
# compute AFE and WBIC for many samples, 
# AFE requires sampling with beta=1 while WBIC requires sampling with beta=1/sqrt(n)
# for different n

In [9]:
import pandas as pd
import json
from pathlib import Path

wbic_bias_filename = "wbic_bias"
wbic_bias_file = Path(f"{outputdir}/{wbic_bias_filename}.csv")
if dry:
  wbic_bias_file = Path(f"{wbic_bias_file}.dry")

# Load existing results if file exists
if wbic_bias_file.exists():
  wbic_bias_df = pd.read_csv(wbic_bias_file)
  # Create a set of completed (run, regime, dsid) tuples for fast lookup
  completed = set(
    zip(wbic_bias_df["trial"], wbic_bias_df["regime"], wbic_bias_df["dsid"], wbic_bias_df["n_components"])
  )
  wbic_bias_data = wbic_bias_df.to_dict("records")
else:
  completed = set()
  wbic_bias_data = []

def save_results():
  pd.DataFrame(wbic_bias_data).to_csv(wbic_bias_file, index=False)

In [12]:
from joblib import Parallel, delayed
from tqdm import tqdm
from itertools import product
import time
from pymc_extensions.tempered_mixpoisson import TemperedPoissonMixture
from pymc_extensions import pmx
from scipy_extensions import mixpoisson
from tqdm.notebook import tqdm
import pymc as pm
import numpy as np
import arviz as az

def run_single_dsid(dsid, n_components, run, regime, datadir, n_draws, n_tune, n_chains):
  dataset = pd.read_csv(f"{datadir}/{dsid}-{regime}.csv")
  X = dataset.iloc[:, run].to_numpy()
  n_obs = len(X)

  afe0 = afe_by_dsid(dsid=dsid, X=X, k=n_components)
  
  with TemperedPoissonMixture(X=X, beta=1/np.log(n_obs), n_components=n_components) as model:
    idata = model.sample(
      draws=n_draws,
      tune=n_tune,
      chains=n_chains,
      progressbar=False,
      nuts_sampler="nutpie" if not dry else "numpyro",
      cores=1
    )
    
    diverging = idata.sample_stats.diverging.values
    divs_per_chain = diverging.sum(axis=1)

    wbic = model.wbic(idata)

    print(f"run={run}, n_components={n_components}, regime={regime}, dsid={dsid}, wbic={wbic:.4f}, afe0={afe0:.4f}")
    
    return {
      "dsid": dsid,
      "regime": regime,
      "n": regime,
      "trial": run,
      "wbic": wbic,
      "afe0": afe0,
      "chains": n_chains,
      "draws": n_draws,
      "tune": n_tune,
      "mean_divergences": divs_per_chain.mean(),
      "total_divergences": diverging.sum(),
      "max_divergences": divs_per_chain.max(),
      "divergences_per_chain": divs_per_chain.tolist(),
      "chain_tree_depth": idata.sample_stats.depth.values.max(),
      "n_components": n_components
    }

# n_cores = os.cpu_count()
# All dsids
all_dsids = dgps["dsid"].unique()

# Group tasks by regime
# if n_cores > len(all_dsids):
#   block_size = int(n_cores/len(all_dsids)-1)
# else:
# block_size = 1

# Main loop - one run at a time, parallelize all (regime, dsid) combos
for run in tqdm(range(0, 1000), desc="runs"):
  start = time.perf_counter()
  # Build list of all (regime, dsid) pairs that haven't been completed
  tasks_to_run = [
    (run, regime, n_components, dsid) 
    for regime in regimes
    for n_components in n_components_list
    for dsid in all_dsids
    if (run, regime, dsid, n_components) not in completed
  ]

  if not tasks_to_run:
    print(f"skipped run={run}")
    continue

  print(f"about to run {tasks_to_run}")

  # Run all regime x dsid combinations in parallel
  results = Parallel(n_jobs=parallel_n_jobs, verbose=parallel_verbose)(
    delayed(run_single_dsid)(
      dsid, n_components, run, regime, datadir, n_draws, n_tune, n_chains
    )
    for run, regime, n_components, dsid in tasks_to_run
  )

  # Collect results
  for result in results:
    wbic_bias_data.append(result)
    completed.add((result["trial"], result["regime"], result["dsid"], result["n_components"]))
  
  end = time.perf_counter()
  # Save after each run completes
  save_results()
  if not dry:
    !git add "../../outputs/mixture/mixpoisson/wbic/wbic_bias.csv"
    !git commit -m f"run {run} complete"
    !git push

runs:   0%|          | 0/1000 [00:00<?, ?it/s]

skipped run=0
about to run [(1, 50, 2, 'regular'), (1, 50, 2, 'e-singular'), (1, 50, 2, 'singular1'), (1, 50, 2, 'singular2'), (1, 50, 3, 'regular'), (1, 50, 3, 'e-singular'), (1, 50, 3, 'singular1'), (1, 50, 3, 'singular2'), (1, 50, 5, 'regular'), (1, 50, 5, 'e-singular'), (1, 50, 5, 'singular1'), (1, 50, 5, 'singular2'), (1, 250, 2, 'regular'), (1, 250, 2, 'e-singular'), (1, 250, 2, 'singular1'), (1, 250, 2, 'singular2'), (1, 250, 3, 'regular'), (1, 250, 3, 'e-singular'), (1, 250, 3, 'singular1'), (1, 250, 3, 'singular2'), (1, 250, 5, 'regular'), (1, 250, 5, 'e-singular'), (1, 250, 5, 'singular1'), (1, 250, 5, 'singular2'), (1, 5000, 2, 'regular'), (1, 5000, 2, 'e-singular'), (1, 5000, 2, 'singular1'), (1, 5000, 2, 'singular2'), (1, 5000, 3, 'regular'), (1, 5000, 3, 'e-singular'), (1, 5000, 3, 'singular1'), (1, 5000, 3, 'singular2'), (1, 5000, 5, 'regular'), (1, 5000, 5, 'e-singular'), (1, 5000, 5, 'singular1'), (1, 5000, 5, 'singular2')]


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
/Users/ashrafahmed/workspace/waterloo-slt-reading-group/zoo/python/libs/scipy_extensions/src/scipy_extensions/mixpoisson.py:21: RuntimeWarning: divide by zero encountered in log
  return logsumexp(result + np.log(weights), axis=1)


KeyboardInterrupt: 